<a href="https://colab.research.google.com/github/Segn11/dataset-analysis-and-manipulation/blob/prediction/loan_default_challenge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [54]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [55]:
import zipfile
import os

zip_file_path = '/content/drive/MyDrive/data-science-nigeria-challenge-1-loan-default-prediction20250307-26022-im3qg9.zip'
extraction_path = '/content/loan_default_data/'

# Create the extraction directory if it doesn't exist
os.makedirs(extraction_path, exist_ok=True)

with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
    zip_ref.extractall(extraction_path)

print(f"Files extracted to: {extraction_path}")

# List the extracted files to verify
print("Extracted files:")
for root, dirs, files in os.walk(extraction_path):
    for file in files:
        print(os.path.join(root, file))

Files extracted to: /content/loan_default_data/
Extracted files:
/content/loan_default_data/testperf.csv
/content/loan_default_data/testprevloans.zip
/content/loan_default_data/traindemographics.csv
/content/loan_default_data/SampleSubmission.csv
/content/loan_default_data/trainperf.csv
/content/loan_default_data/manifest-155273bc0bf3f6c96da245fb19b9dc9320250307-26022-7yk61d.json
/content/loan_default_data/testdemographics.csv
/content/loan_default_data/trainprevloans.zip


In [56]:
import pandas as pd
import zipfile
import os

In [57]:
train0 = pd.read_csv('/content/loan_default_data/trainperf.csv')
train1 = pd.read_csv('/content/loan_default_data/traindemographics.csv')
train2 = pd.read_csv('/content/loan_default_data/trainprevloans.zip')

In [58]:
train0.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4368 entries, 0 to 4367
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   customerid     4368 non-null   object 
 1   systemloanid   4368 non-null   int64  
 2   loannumber     4368 non-null   int64  
 3   approveddate   4368 non-null   object 
 4   creationdate   4368 non-null   object 
 5   loanamount     4368 non-null   float64
 6   totaldue       4368 non-null   float64
 7   termdays       4368 non-null   int64  
 8   referredby     587 non-null    object 
 9   good_bad_flag  4368 non-null   object 
dtypes: float64(2), int64(3), object(5)
memory usage: 341.4+ KB


In [59]:
test0 = pd.read_csv('/content/loan_default_data/testperf.csv')
test1 = pd.read_csv('/content/loan_default_data/testprevloans.zip')
test2 = pd.read_csv('/content/loan_default_data/testdemographics.csv')

In [60]:
data_path = '/content/loan_default_data/'

# Load testdemographics.csv
test2 = pd.read_csv(os.path.join(data_path, 'testdemographics.csv'))
print("Test Demographics Data (Head):")
display(test2.head())
print("\nTest Demographics Info:")
test2.info()

Test Demographics Data (Head):


,customerid,birthdate,bank_account_type,longitude_gps,latitude_gps,bank_name_clients,bank_branch_clients,employment_status_clients,level_of_education_clients
0,8a858f305c8dd672015c93b1db645db4,1976-08-28 00:00:00.000000,Savings,5.296628,7.593965,Heritage Bank,NaN,Permanent,NaN
1,8a858f085a477386015a47fb049e49ca,1978-06-23 00:00:00.000000,Savings,3.294513,6.596602,UBA,NaN,Permanent,NaN
2,8a858e6f5cd5e874015cd6f5634c39ad,1984-04-04 00:00:00.000000,Savings,8.501912,7.729364,First Bank,NaN,Permanent,NaN
3,8a858e9d5bfd7037015bfdab79f61305,1983-05-28 00:00:00.000000,Savings,3.318904,6.681595,UBA,NaN,Permanent,NaN
4,8a858fde56eb02280156eb6dafc128ac,1982-03-29 00:00:00.000000,Savings,6.354624,4.949031,First Bank,NaN,Self-Employed,NaN



Test Demographics Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1487 entries, 0 to 1486
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   customerid                  1487 non-null   object 
 1   birthdate                   1487 non-null   object 
 2   bank_account_type           1487 non-null   object 
 3   longitude_gps               1487 non-null   float64
 4   latitude_gps                1487 non-null   float64
 5   bank_name_clients           1487 non-null   object 
 6   bank_branch_clients         14 non-null     object 
 7   employment_status_clients   1270 non-null   object 
 8   level_of_education_clients  210 non-null    object 
dtypes: float64(2), object(7)
memory usage: 104.7+ KB


Now, let's merge the `test2` (demographics) with the existing `test` dataframe.

In [61]:
# First, ensure test_prevloans_agg is created by aggregating test1
test_prevloans_agg = test1.groupby('customerid').agg(
    num_prev_loans=('systemloanid', 'count'),
    total_prev_loanamount=('loanamount', 'sum'),
    avg_prev_loanamount=('loanamount', 'mean'),
    total_prev_totaldue=('totaldue', 'sum'),
    avg_prev_totaldue=('totaldue', 'mean'),
    avg_prev_termdays=('termdays', 'mean')
).reset_index()

# Then, create the initial 'test' dataframe by merging test0 with test_prevloans_agg
test = pd.merge(test0, test_prevloans_agg, on='customerid', how='left')

# Now, merge this 'test' dataframe with 'test2' (demographics)
# We'll use 'customerid' as the common key
test = pd.merge(test, test2, on='customerid', how='left')

print("\nFinal Merged Test Data (Head - with demographics):")
display(test.head())
print("\nFinal Merged Test Data Info - with demographics:")
test.info()
print("\nShape of the final test dataset (with demographics):", test.shape)


Final Merged Test Data (Head - with demographics):


,customerid,systemloanid,loannumber,approveddate,creationdate,loanamount,totaldue,termdays,referredby,num_prev_loans,...,avg_prev_totaldue,avg_prev_termdays,birthdate,bank_account_type,longitude_gps,latitude_gps,bank_name_clients,bank_branch_clients,employment_status_clients,level_of_education_clients
0,8a858899538ddb8e015390510b321f08,301998974,4,40:48.0,39:35.0,10000,12250.0,30,NaN,3.0,...,10966.666667,25.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,8a858959537a097401537a4e316e25f7,301963615,10,43:40.0,42:34.0,40000,44000.0,30,NaN,9.0,...,27600.000000,31.666667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,8a8589c253ace09b0153af6ba58f1f31,301982236,6,15:11.0,15:04.0,20000,24500.0,30,NaN,5.0,...,15935.000000,27.000000,1981-09-05 00:00:00.000000,Savings,3.227945,6.586668,UBA,NaN,Permanent,NaN
3,8a858e095aae82b7015aae86ca1e030b,301971730,8,00:54.0,00:49.0,30000,34500.0,30,NaN,7.0,...,19342.857143,17.142857,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,8a858e225a28c713015a30db5c48383d,301959177,4,04:33.0,04:27.0,20000,24500.0,30,NaN,3.0,...,12500.000000,25.000000,1975-08-25 00:00:00.000000,Savings,5.248368,13.059864,UBA,NaN,Permanent,NaN



Final Merged Test Data Info - with demographics:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1450 entries, 0 to 1449
Data columns (total 23 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   customerid                  1450 non-null   object 
 1   systemloanid                1450 non-null   int64  
 2   loannumber                  1450 non-null   int64  
 3   approveddate                1450 non-null   object 
 4   creationdate                1450 non-null   object 
 5   loanamount                  1450 non-null   int64  
 6   totaldue                    1450 non-null   float64
 7   termdays                    1450 non-null   int64  
 8   referredby                  184 non-null    object 
 9   num_prev_loans              1442 non-null   float64
 10  total_prev_loanamount       1442 non-null   float64
 11  avg_prev_loanamount         1442 non-null   float64
 12  total_prev_totaldue         1442 non-nul

In [62]:
import zipfile
import os
import pandas as pd

In [63]:
print("Columns in train0:", train0.columns.tolist())
print("Columns in train1:", train1.columns.tolist())
print("Columns in train2:", train2.columns.tolist())

# Merge train0 (performance) and train1 (demographics) first
train = pd.merge(train0, train1, on='customerid', how='left')

print("\nTrain (merged train0 and train1) Head:")
display(train.head())
print("\nTrain (merged train0 and train1) Info:")
train.info()

Columns in train0: ['customerid', 'systemloanid', 'loannumber', 'approveddate', 'creationdate', 'loanamount', 'totaldue', 'termdays', 'referredby', 'good_bad_flag']
Columns in train1: ['customerid', 'birthdate', 'bank_account_type', 'longitude_gps', 'latitude_gps', 'bank_name_clients', 'bank_branch_clients', 'employment_status_clients', 'level_of_education_clients']
Columns in train2: ['customerid', 'systemloanid', 'loannumber', 'approveddate', 'creationdate', 'loanamount', 'totaldue', 'termdays', 'closeddate', 'referredby', 'firstduedate', 'firstrepaiddate']

Train (merged train0 and train1) Head:


,customerid,systemloanid,loannumber,approveddate,creationdate,loanamount,totaldue,termdays,referredby,good_bad_flag,birthdate,bank_account_type,longitude_gps,latitude_gps,bank_name_clients,bank_branch_clients,employment_status_clients,level_of_education_clients
0,8a2a81a74ce8c05d014cfb32a0da1049,301994762,12,2017-07-25 08:22:56.000000,2017-07-25 07:22:47.000000,30000.0,34500.0,30,NaN,Good,1972-01-15 00:00:00.000000,Other,3.432010,6.433055,Diamond Bank,NaN,Permanent,Post-Graduate
1,8a85886e54beabf90154c0a29ae757c0,301965204,2,2017-07-05 17:04:41.000000,2017-07-05 16:04:18.000000,15000.0,17250.0,30,NaN,Good,1985-08-23 00:00:00.000000,Savings,3.885298,7.320700,GT Bank,"DUGBE,IBADAN",Permanent,Graduate
2,8a8588f35438fe12015444567666018e,301966580,7,2017-07-06 14:52:57.000000,2017-07-06 13:52:51.000000,20000.0,22250.0,15,NaN,Good,1984-09-18 00:00:00.000000,Other,11.139350,10.292041,EcoBank,NaN,Permanent,NaN
3,8a85890754145ace015429211b513e16,301999343,3,2017-07-27 19:00:41.000000,2017-07-27 18:00:35.000000,10000.0,11500.0,15,NaN,Good,1977-10-10 00:00:00.000000,Savings,3.985770,7.491708,First Bank,NaN,Permanent,NaN
4,8a858970548359cc0154883481981866,301962360,9,2017-07-03 23:42:45.000000,2017-07-03 22:42:39.000000,40000.0,44000.0,30,NaN,Good,1986-09-07 00:00:00.000000,Other,7.457913,9.076574,GT Bank,NaN,Permanent,Primary



Train (merged train0 and train1) Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4376 entries, 0 to 4375
Data columns (total 18 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   customerid                  4376 non-null   object 
 1   systemloanid                4376 non-null   int64  
 2   loannumber                  4376 non-null   int64  
 3   approveddate                4376 non-null   object 
 4   creationdate                4376 non-null   object 
 5   loanamount                  4376 non-null   float64
 6   totaldue                    4376 non-null   float64
 7   termdays                    4376 non-null   int64  
 8   referredby                  589 non-null    object 
 9   good_bad_flag               4376 non-null   object 
 10  birthdate                   3277 non-null   object 
 11  bank_account_type           3277 non-null   object 
 12  longitude_gps               3277 non-null   float6

Now, let's aggregate `train2` (previous loans) data by `customerid` to avoid duplicating rows when merging. We'll calculate some summary statistics for previous loans.

In [64]:
# Aggregate train2 (previous loans) by customerid
prevloans_agg = train2.groupby('customerid').agg(
    num_prev_loans=('systemloanid', 'count'),
    total_prev_loanamount=('loanamount', 'sum'),
    avg_prev_loanamount=('loanamount', 'mean'),
    total_prev_totaldue=('totaldue', 'sum'),
    avg_prev_totaldue=('totaldue', 'mean'),
    avg_prev_termdays=('termdays', 'mean')
).reset_index()

print("\nAggregated Previous Loans Data (Head):")
display(prevloans_agg.head())
print("\nAggregated Previous Loans Data Info:")
prevloans_agg.info()


Aggregated Previous Loans Data (Head):


,customerid,num_prev_loans,total_prev_loanamount,avg_prev_loanamount,total_prev_totaldue,avg_prev_totaldue,avg_prev_termdays
0,8a1088a0484472eb01484669e3ce4e0b,1,10000.0,10000.000000,11500.0,11500.000000,15.000000
1,8a1a1e7e4f707f8b014f797718316cad,4,70000.0,17500.000000,89500.0,22375.000000,37.500000
2,8a1a32fc49b632520149c3b8fdf85139,7,90000.0,12857.142857,106500.0,15214.285714,19.285714
3,8a1eb5ba49a682300149c3c068b806c7,8,130000.0,16250.000000,162400.0,20300.000000,33.750000
4,8a1edbf14734127f0147356fdb1b1eb2,2,20000.0,10000.000000,24500.0,12250.000000,22.500000



Aggregated Previous Loans Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4359 entries, 0 to 4358
Data columns (total 7 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   customerid             4359 non-null   object 
 1   num_prev_loans         4359 non-null   int64  
 2   total_prev_loanamount  4359 non-null   float64
 3   avg_prev_loanamount    4359 non-null   float64
 4   total_prev_totaldue    4359 non-null   float64
 5   avg_prev_totaldue      4359 non-null   float64
 6   avg_prev_termdays      4359 non-null   float64
dtypes: float64(5), int64(1), object(1)
memory usage: 238.5+ KB


Finally, merge the aggregated previous loans data (`prevloans_agg`) with the combined `train` dataset.

In [65]:
# Merge the aggregated previous loans with the main train dataframe
train = pd.merge(train, prevloans_agg, on='customerid', how='left')

print("\nFinal Merged Train Data (Head):")
display(train.head())
print("\nFinal Merged Train Data Info:")
train.info()
print("\nShape of the final train dataset:", train.shape)


Final Merged Train Data (Head):


,customerid,systemloanid,loannumber,approveddate,creationdate,loanamount,totaldue,termdays,referredby,good_bad_flag,...,bank_name_clients,bank_branch_clients,employment_status_clients,level_of_education_clients,num_prev_loans,total_prev_loanamount,avg_prev_loanamount,total_prev_totaldue,avg_prev_totaldue,avg_prev_termdays
0,8a2a81a74ce8c05d014cfb32a0da1049,301994762,12,2017-07-25 08:22:56.000000,2017-07-25 07:22:47.000000,30000.0,34500.0,30,NaN,Good,...,Diamond Bank,NaN,Permanent,Post-Graduate,11.0,200000.0,18181.818182,242900.0,22081.818182,30.0
1,8a85886e54beabf90154c0a29ae757c0,301965204,2,2017-07-05 17:04:41.000000,2017-07-05 16:04:18.000000,15000.0,17250.0,30,NaN,Good,...,GT Bank,"DUGBE,IBADAN",Permanent,Graduate,NaN,NaN,NaN,NaN,NaN,NaN
2,8a8588f35438fe12015444567666018e,301966580,7,2017-07-06 14:52:57.000000,2017-07-06 13:52:51.000000,20000.0,22250.0,15,NaN,Good,...,EcoBank,NaN,Permanent,NaN,6.0,60000.0,10000.000000,70500.0,11750.000000,17.5
3,8a85890754145ace015429211b513e16,301999343,3,2017-07-27 19:00:41.000000,2017-07-27 18:00:35.000000,10000.0,11500.0,15,NaN,Good,...,First Bank,NaN,Permanent,NaN,2.0,20000.0,10000.000000,24500.0,12250.000000,22.5
4,8a858970548359cc0154883481981866,301962360,9,2017-07-03 23:42:45.000000,2017-07-03 22:42:39.000000,40000.0,44000.0,30,NaN,Good,...,GT Bank,NaN,Permanent,Primary,8.0,150000.0,18750.000000,188400.0,23550.000000,37.5



Final Merged Train Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4376 entries, 0 to 4375
Data columns (total 24 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   customerid                  4376 non-null   object 
 1   systemloanid                4376 non-null   int64  
 2   loannumber                  4376 non-null   int64  
 3   approveddate                4376 non-null   object 
 4   creationdate                4376 non-null   object 
 5   loanamount                  4376 non-null   float64
 6   totaldue                    4376 non-null   float64
 7   termdays                    4376 non-null   int64  
 8   referredby                  589 non-null    object 
 9   good_bad_flag               4376 non-null   object 
 10  birthdate                   3277 non-null   object 
 11  bank_account_type           3277 non-null   object 
 12  longitude_gps               3277 non-null   float64
 13  la

In [66]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4376 entries, 0 to 4375
Data columns (total 24 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   customerid                  4376 non-null   object 
 1   systemloanid                4376 non-null   int64  
 2   loannumber                  4376 non-null   int64  
 3   approveddate                4376 non-null   object 
 4   creationdate                4376 non-null   object 
 5   loanamount                  4376 non-null   float64
 6   totaldue                    4376 non-null   float64
 7   termdays                    4376 non-null   int64  
 8   referredby                  589 non-null    object 
 9   good_bad_flag               4376 non-null   object 
 10  birthdate                   3277 non-null   object 
 11  bank_account_type           3277 non-null   object 
 12  longitude_gps               3277 non-null   float64
 13  latitude_gps                3277 

In [67]:
test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1450 entries, 0 to 1449
Data columns (total 23 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   customerid                  1450 non-null   object 
 1   systemloanid                1450 non-null   int64  
 2   loannumber                  1450 non-null   int64  
 3   approveddate                1450 non-null   object 
 4   creationdate                1450 non-null   object 
 5   loanamount                  1450 non-null   int64  
 6   totaldue                    1450 non-null   float64
 7   termdays                    1450 non-null   int64  
 8   referredby                  184 non-null    object 
 9   num_prev_loans              1442 non-null   float64
 10  total_prev_loanamount       1442 non-null   float64
 11  avg_prev_loanamount         1442 non-null   float64
 12  total_prev_totaldue         1442 non-null   float64
 13  avg_prev_totaldue           1442 

In [68]:
sampsub = pd.read_csv('/content/loan_default_data/SampleSubmission.csv')
#sampsub.head()

In [69]:
test_id = test.drop(['customerid'], axis=1)
train_id = train.drop(['customerid', 'systemloanid'], axis=1)

test_is = test.drop(['systemloanid'], axis=1)


In [70]:
cat_cols = train.select_dtypes(include="object").columns
cat_cols = list(cat_cols)
print(cat_cols)

['customerid', 'approveddate', 'creationdate', 'referredby', 'good_bad_flag', 'birthdate', 'bank_account_type', 'bank_name_clients', 'bank_branch_clients', 'employment_status_clients', 'level_of_education_clients']


In [71]:
num_cols = test.select_dtypes(include="number").columns
num_cols = list(num_cols)
print(num_cols)

['systemloanid', 'loannumber', 'loanamount', 'totaldue', 'termdays', 'num_prev_loans', 'total_prev_loanamount', 'avg_prev_loanamount', 'total_prev_totaldue', 'avg_prev_totaldue', 'avg_prev_termdays', 'longitude_gps', 'latitude_gps']


In [72]:
# Select numeric columns dynamically
num_cols_train = train.select_dtypes(include=['int64', 'float64']).columns
num_cols_test = test.select_dtypes(include=['int64', 'float64']).columns

# Select categorical columns dynamically
cat_cols_train = train.select_dtypes(include='object').columns
cat_cols_test = test.select_dtypes(include='object').columns

# Fill missing values
train[num_cols_train] = train[num_cols_train].fillna(0)
test[num_cols_test] = test[num_cols_test].fillna(0)

train[cat_cols_train] = train[cat_cols_train].fillna('unknown')
test[cat_cols_test] = test[cat_cols_test].fillna('unknown')

In [73]:
# Convert to datetime
train['approveddate'] = pd.to_datetime(train['approveddate'], errors='coerce')
train['creationdate'] = pd.to_datetime(train['creationdate'], errors='coerce')

test['approveddate'] = pd.to_datetime(test['approveddate'], errors='coerce')
test['creationdate'] = pd.to_datetime(test['creationdate'], errors='coerce')


/tmp/ipython-input-1897446360.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  test['approveddate'] = pd.to_datetime(test['approveddate'], errors='coerce')
/tmp/ipython-input-1897446360.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  test['creationdate'] = pd.to_datetime(test['creationdate'], errors='coerce')


In [74]:
# Year, month, day, weekday
for df in [train, test]:
    df['approved_year'] = df['approveddate'].dt.year
    df['approved_month'] = df['approveddate'].dt.month
    df['approved_day'] = df['approveddate'].dt.day
    df['approved_weekday'] = df['approveddate'].dt.weekday  # 0=Monday, 6=Sunday

    # Loan age in days
    df['loan_age_days'] = (df['approveddate'] - df['creationdate']).dt.days
train = train.drop(['approveddate', 'creationdate'], axis=1)
test = test.drop(['approveddate', 'creationdate'], axis=1)
# Select numeric columns dynamically
num_cols_train = train.select_dtypes(include=['int64', 'float64']).columns
num_cols_test = test.select_dtypes(include=['int64', 'float64']).columns

# Select categorical columns dynamically
cat_cols_train = train.select_dtypes(include='object').columns
cat_cols_test = test.select_dtypes(include='object').columns

# Fill missing values
train[num_cols_train] = train[num_cols_train].fillna(0)
test[num_cols_test] = test[num_cols_test].fillna(0)

train[cat_cols_train] = train[cat_cols_train].fillna('unknown')
test[cat_cols_test] = test[cat_cols_test].fillna('unknown')
today = pd.Timestamp.today()
train['age'] = (today - pd.to_datetime(train['birthdate'], errors='coerce')).dt.days // 365
test['age'] = (today - pd.to_datetime(test['birthdate'], errors='coerce')).dt.days // 365

train.drop('birthdate', axis=1, inplace=True)
test.drop('birthdate', axis=1, inplace=True)

# Select numeric columns dynamically
num_cols_train = train.select_dtypes(include=['int64', 'float64']).columns
num_cols_test = test.select_dtypes(include=['int64', 'float64']).columns

# Select categorical columns dynamically
cat_cols_train = train.select_dtypes(include='object').columns
cat_cols_test = test.select_dtypes(include='object').columns

# Fill missing values
train[num_cols_train] = train[num_cols_train].fillna(0)
test[num_cols_test] = test[num_cols_test].fillna(0)

train[cat_cols_train] = train[cat_cols_train].fillna('unknown')
test[cat_cols_test] = test[cat_cols_test].fillna('unknown')


test_ids = test['customerid'].copy()


/tmp/ipython-input-93931821.py:28: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  test['age'] = (today - pd.to_datetime(test['birthdate'], errors='coerce')).dt.days // 365


In [75]:
# Target variable
# Check if 'good_bad_flag' exists before extracting and converting
if 'good_bad_flag' in train.columns:
    y = train['good_bad_flag'].map({'Bad': 0, 'Good': 1})
    # Drop 'good_bad_flag' from train now that y is set
    train = train.drop('good_bad_flag', axis=1)
else:
    print("Warning: 'good_bad_flag' column not found in 'train' DataFrame. Assuming 'y' is already set.")

# Drop other ID columns from train
cols_to_drop_train_ids = ['customerid', 'systemloanid']
for col in cols_to_drop_train_ids:
    if col in train.columns:
        train = train.drop(col, axis=1)

# Drop ID columns from test
cols_to_drop_test_ids = ['customerid', 'systemloanid']
for col in cols_to_drop_test_ids:
    if col in test.columns:
        test = test.drop(col, axis=1)

In [76]:
# Define the mapping for education
edu_map = {
    "No formal education": 0,
    "Primary education": 1,
    "Secondary education": 2,
    "Tertiary education": 3
}

# Encode train
train["education_num"] = train["level_of_education_clients"].map(edu_map).fillna(-1).astype(int)

# Encode test
test["education_num"] = test["level_of_education_clients"].map(edu_map).fillna(-1).astype(int)

# Drop the original categorical column
train = train.drop("level_of_education_clients", axis=1)
test = test.drop("level_of_education_clients", axis=1)



In [77]:
train["loanamount_per_age"] = train["loanamount"] / (train["age"] + 1)

test["loanamount_per_age"] = test["loanamount"] / (test["age"] + 1)

train["totaldue_per_age"] = train["totaldue"] / (train["age"] + 1)

test["totaldue_per_age"] = test["totaldue"] / (test["age"] + 1)

train["loanamount_per_age"].isna().sum()
test["loanamount_per_age"].isna().sum()

train["totaldue_per_age"].isna().sum()
test["totaldue_per_age"].isna().sum()

np.int64(0)

In [78]:
from sklearn.preprocessing import OrdinalEncoder

# --- Ordinal features ---
ordinal_cols = [ 'employment_status_clients']  # make sure 'education_num' already exists
train[ordinal_cols] = train[ordinal_cols].fillna(0)  # handle missing values
test[ordinal_cols] = test[ordinal_cols].fillna(0)

ord_enc = OrdinalEncoder()
train[ordinal_cols] = ord_enc.fit_transform(train[ordinal_cols])
test[ordinal_cols] = ord_enc.transform(test[ordinal_cols])

# --- One-hot features ---
onehot_cols = ['bank_account_type', 'bank_name_clients', 'bank_branch_clients', 'referredby']

# Fill missing values
train[onehot_cols] = train[onehot_cols].fillna('unknown')
test[onehot_cols] = test[onehot_cols].fillna('unknown')

# One-hot encoding
train = pd.get_dummies(train, columns=onehot_cols, prefix_sep="_")
test = pd.get_dummies(test, columns=onehot_cols, prefix_sep="_")

# --- Align train/test columns ---
train, test = train.align(test, join='left', axis=1, fill_value=0)

In [79]:
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(train, y, test_size=0.2, random_state=42, stratify=y)

print("Training and validation sets created successfully.")
print(f"X_train shape: {X_train.shape}")
print(f"X_val shape: {X_val.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_val shape: {y_val.shape}")

Training and validation sets created successfully.
X_train shape: (3500, 599)
X_val shape: (876, 599)
y_train shape: (3500,)
y_val shape: (876,)


In [80]:
from sklearn.model_selection import train_test_split
X_train, X_val, y_train, y_val = train_test_split(train, y, test_size=0.2, random_state=42, stratify=y)

print("Training and validation sets created successfully.")
print(f"X_train shape: {X_train.shape}")
print(f"X_val shape: {X_val.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_val shape: {y_val.shape}")

Training and validation sets created successfully.
X_train shape: (3500, 599)
X_val shape: (876, 599)
y_train shape: (3500,)
y_val shape: (876,)


In [81]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train, y_train)


# Initialize StandardScaler
scaler = StandardScaler()

# Fit the scaler on X_train and transform both X_train and X_val
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)



/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [82]:
# Initialize and train the Logistic Regression model with scaled data
model = LogisticRegression(max_iter=1000, random_state=42)
model.fit(X_train_scaled, y_train)
print("Logistic Regression model initialized and trained successfully with scaled data.")
y_pred = model.predict(X_val_scaled)
y_prob = model.predict_proba(X_val_scaled)[:, 1]

Logistic Regression model initialized and trained successfully with scaled data.


In [83]:
from sklearn.metrics import accuracy_score
accuracy = accuracy_score(y_val, y_pred)
error_rate = (1 - accuracy) * 100

print(f"Accuracy Score: {accuracy:.4f}")
print(f"Error Rate: {error_rate:.2f}%")

Accuracy Score: 0.7808
Error Rate: 21.92%


In [84]:
# Scale the test data using the same scaler fitted on the training data
X_test_scaled = scaler.transform(test)

# Make predictions on the scaled test data
y_test_pred = model.predict(X_test_scaled)
y_test_prob = model.predict_proba(X_test_scaled)[:, 1]

print("Predictions on the test set completed successfully.")
print("Sample of predicted classes (first 10):", y_test_pred[:10])
print("Sample of predicted probabilities for positive class (first 10):", y_test_prob[:10])

Predictions on the test set completed successfully.
Sample of predicted classes (first 10): [1 1 1 1 1 1 1 1 1 1]
Sample of predicted probabilities for positive class (first 10): [0.71187939 0.9401247  0.83638701 0.88618983 0.81315039 0.87547426
 0.78734945 0.75510228 0.8112519  0.97710299]


In [85]:
submission = pd.DataFrame({
    'customerID': test_ids,
    'Good_Bad_flag': y_test_pred
})

#submission.to_csv("submission.csv", index=False)

print("Submission preview:")
print(submission.head())

Submission preview:
                         customerID  Good_Bad_flag
0  8a858899538ddb8e015390510b321f08              1
1  8a858959537a097401537a4e316e25f7              1
2  8a8589c253ace09b0153af6ba58f1f31              1
3  8a858e095aae82b7015aae86ca1e030b              1
4  8a858e225a28c713015a30db5c48383d              1


# Task
Optimize the Logistic Regression model for loan default prediction by applying Recursive Feature Elimination (RFE) to select the most important numerical features, re-training the model on the refined feature set, and generating a submission file with the improved predictions. Also, summarize the performance improvement.

## Apply RFE Feature Selection

### Subtask:
Apply Recursive Feature Elimination (RFE) using a Logistic Regression estimator to select the most important features from the numerical training data. This will help reduce dimensionality and potentially improve model performance.


**Reasoning**:
To apply Recursive Feature Elimination, I need to import the necessary classes and initialize the RFE model with a Logistic Regression estimator, then fit it to the training data.



In [86]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

# Initialize a Logistic Regression model as the estimator for RFE
# Ensure max_iter and random_state match previous model parameters
estimator = LogisticRegression(max_iter=1000, random_state=42)

# Initialize RFE with the estimator, specifying n_features_to_select and step
# n_features_to_select is set to 50 as per instruction (or another appropriate number)
# step=1 means to remove one feature at each iteration
rfe_selector = RFE(estimator=estimator, n_features_to_select=50, step=1)

# Fit the RFE model to the training data
rfe_selector.fit(X_train_scaled, y_train)

print("RFE selector fitted successfully.")

RFE selector fitted successfully.


**Reasoning**:
Now that RFE has been fitted, I need to extract the selected features from the training, validation, and test datasets. This will create new datasets with only the most important features as determined by RFE.



In [87]:
X_train_selected = X_train_scaled[:, rfe_selector.support_]
X_val_selected = X_val_scaled[:, rfe_selector.support_]
X_test_selected = X_test_scaled[:, rfe_selector.support_]

print(f"Original X_train_scaled shape: {X_train_scaled.shape}")
print(f"X_train_selected shape: {X_train_selected.shape}")
print(f"Original X_val_scaled shape: {X_val_scaled.shape}")
print(f"X_val_selected shape: {X_val_selected.shape}")
print(f"Original X_test_scaled shape: {X_test_scaled.shape}")
print(f"X_test_selected shape: {X_test_selected.shape}")

Original X_train_scaled shape: (3500, 599)
X_train_selected shape: (3500, 50)
Original X_val_scaled shape: (876, 599)
X_val_selected shape: (876, 50)
Original X_test_scaled shape: (1450, 599)
X_test_selected shape: (1450, 50)


## Re-train the Model with Selected Features

### Subtask:
Re-train the Logistic Regression model using the features selected by RFE (`X_train_selected`, `X_val_selected`), and then evaluate its performance on the validation set.

**Reasoning**:
Now that the features have been selected using RFE, I need to re-initialize and train a new Logistic Regression model using only these selected features. This will allow for evaluation of the model's performance with the reduced feature set.



In [88]:
from sklearn.linear_model import LogisticRegression

# Initialize a new Logistic Regression model with the same parameters
model_rfe = LogisticRegression(max_iter=1000, random_state=42)

# Train the model using the RFE-selected scaled training data
model_rfe.fit(X_train_selected, y_train)

print("Logistic Regression model re-trained successfully with RFE-selected features.")

# Make predictions on the RFE-selected scaled validation data
y_pred_rfe = model_rfe.predict(X_val_selected)
y_prob_rfe = model_rfe.predict_proba(X_val_selected)[:, 1]

print("Predictions on validation set with RFE-selected features completed.")

Logistic Regression model re-trained successfully with RFE-selected features.
Predictions on validation set with RFE-selected features completed.


**Reasoning**:
To evaluate the performance of the Logistic Regression model after RFE feature selection, I need to calculate the accuracy score and error rate on the validation set, comparing it to the performance of the original model.



In [89]:
from sklearn.metrics import accuracy_score

accuracy_rfe = accuracy_score(y_val, y_pred_rfe)
error_rate_rfe = (1 - accuracy_rfe) * 100

print(f"Accuracy Score with RFE: {accuracy_rfe:.4f}")
print(f"Error Rate with RFE: {error_rate_rfe:.2f}%")

print("\n--- Comparison with Original Model ---")
print(f"Original Accuracy Score: {accuracy:.4f}")
print(f"Original Error Rate: {error_rate:.2f}%")

performance_summary = {
    "original_accuracy": accuracy,
    "original_error_rate": error_rate,
    "rfe_accuracy": accuracy_rfe,
    "rfe_error_rate": error_rate_rfe
}

Accuracy Score with RFE: 0.7808
Error Rate with RFE: 21.92%

--- Comparison with Original Model ---
Original Accuracy Score: 0.7808
Original Error Rate: 21.92%


**Reasoning**:
To complete the subtask of generating a submission file, I need to use the RFE-trained model (`model_rfe`) to make predictions on the RFE-selected test data (`X_test_selected`). These predictions will then be formatted into a submission DataFrame using the previously saved `test_ids`.



In [90]:
y_test_pred_rfe = model_rfe.predict(X_test_selected)
y_test_prob_rfe = model_rfe.predict_proba(X_test_selected)[:, 1]

submission_rfe = pd.DataFrame({
    'customerID': test_ids,
    'Good_Bad_flag': y_test_pred_rfe
})

print("Submission preview with RFE-selected features:")
print(submission_rfe.head())

# Save the submission file if needed
# submission_rfe.to_csv("submission_rfe.csv", index=False)

Submission preview with RFE-selected features:
                         customerID  Good_Bad_flag
0  8a858899538ddb8e015390510b321f08              1
1  8a858959537a097401537a4e316e25f7              1
2  8a8589c253ace09b0153af6ba58f1f31              1
3  8a858e095aae82b7015aae86ca1e030b              1
4  8a858e225a28c713015a30db5c48383d              1


### Performance Improvement Summary

- **Original Model Accuracy**: 0.7808
- **Original Model Error Rate**: 21.92%

- **RFE Model Accuracy**: 0.7808
- **RFE Model Error Rate**: 21.92%

In this instance, applying RFE with `n_features_to_select=50` did not lead to a change in the accuracy score or error rate on the validation set compared to the full feature set. While accuracy remained the same, RFE successfully reduced the dimensionality of the feature space from 599 features to 50, which can lead to benefits such as faster training times and potentially better generalization on unseen data, even if the validation accuracy didn't immediately reflect an improvement.

### Performance Improvement Summary

- **Original Model Accuracy**: 0.7808
- **Original Model Error Rate**: 21.92%

- **RFE Model Accuracy**: 0.7808
- **RFE Model Error Rate**: 21.92%

In this instance, applying RFE with `n_features_to_select=50` did not lead to a change in the accuracy score or error rate on the validation set compared to the full feature set. While accuracy remained the same, RFE successfully reduced the dimensionality of the feature space from 599 features to 50, which can lead to benefits such as faster training times and potentially better generalization on unseen data, even if the validation accuracy didn't immediately reflect an improvement.

## Final Task

### Subtask:
Summarize the optimization process and the performance of the Logistic Regression model after applying RFE.


## Summary:

### Q&A
The optimization process involved applying Recursive Feature Elimination (RFE) to select the 50 most important features from the original 599 numerical features, using a Logistic Regression estimator. The model was then re-trained on this reduced feature set. After applying RFE, the Logistic Regression model achieved an accuracy of 0.7808 and an error rate of 21.92% on the validation set. This performance was identical to the original model's accuracy and error rate.

### Data Analysis Key Findings
*   Recursive Feature Elimination (RFE) successfully reduced the feature space from 599 to 50 features across the training, validation, and test datasets.
*   A Logistic Regression model re-trained on these 50 RFE-selected features achieved an accuracy of 0.7808 and an error rate of 21.92% on the validation set.
*   Compared to the original model, which also had an accuracy of 0.7808 and an error rate of 21.92%, RFE did not lead to an improvement in accuracy on the validation set for this specific configuration.
*   Despite no direct accuracy improvement, the model complexity was significantly reduced by using only 50 features instead of 599.

### Insights or Next Steps
*   While validation accuracy remained unchanged, the significant reduction in features (from 599 to 50) due to RFE is beneficial for model interpretability, reduced training time, and potential for better generalization on unseen data, even if not immediately reflected in validation accuracy.
*   To potentially improve performance, consider experimenting with different numbers of features to select (`n_features_to_select`) in RFE, or explore alternative feature selection techniques and hyperparameter tuning for the Logistic Regression model.


# Task
Apply Recursive Feature Elimination with Cross-Validation (RFECV) using a Logistic Regression estimator on the scaled training data (`X_train_scaled` and `y_train`) to find the optimal number of features.

## Apply RFECV for Optimal Feature Selection

### Subtask:
Apply Recursive Feature Elimination with Cross-Validation (RFECV) using a Logistic Regression estimator to identify the optimal number of features that yield the best performance, as determined by cross-validation. This will automatically select a more suitable number of features than a fixed 'n_features_to_select'.


**Reasoning**:
To perform Recursive Feature Elimination with Cross-Validation (RFECV) for optimal feature selection, I need to import the necessary classes, initialize the Logistic Regression estimator, and then initialize and fit the RFECV selector to the scaled training data. This will determine the optimal number of features.



In [91]:
from sklearn.feature_selection import RFECV
from sklearn.linear_model import LogisticRegression

# Initialize a Logistic Regression model as the estimator for RFECV
estimator = LogisticRegression(max_iter=1000, random_state=42)

# Initialize RFECV with the estimator, specifying step and cross-validation folds
# scoring='accuracy' is a common choice for classification tasks
rfecv_selector = RFECV(estimator=estimator, step=1, cv=5, scoring='accuracy')

# Fit the RFECV model to the scaled training data
rfecv_selector.fit(X_train_scaled, y_train)

print("RFECV selector fitted successfully.")
print(f"Optimal number of features: {rfecv_selector.n_features_}")

RFECV selector fitted successfully.
Optimal number of features: 15


**Reasoning**:
Now that RFECV has been fitted and the optimal number of features identified, I need to extract these selected features from the scaled training, validation, and test datasets. This will create new datasets with only the most important features as determined by RFECV.



In [92]:
X_train_rfecv = X_train_scaled[:, rfecv_selector.support_]
X_val_rfecv = X_val_scaled[:, rfecv_selector.support_]
X_test_rfecv = X_test_scaled[:, rfecv_selector.support_]

print(f"Original X_train_scaled shape: {X_train_scaled.shape}")
print(f"X_train_rfecv shape: {X_train_rfecv.shape}")
print(f"Original X_val_scaled shape: {X_val_scaled.shape}")
print(f"X_val_rfecv shape: {X_val_rfecv.shape}")
print(f"Original X_test_scaled shape: {X_test_scaled.shape}")
print(f"X_test_rfecv shape: {X_test_rfecv.shape}")

Original X_train_scaled shape: (3500, 599)
X_train_rfecv shape: (3500, 15)
Original X_val_scaled shape: (876, 599)
X_val_rfecv shape: (876, 15)
Original X_test_scaled shape: (1450, 599)
X_test_rfecv shape: (1450, 15)


## Re-scale Data with RFECV Selected Features

### Subtask:
Re-scale the features selected by RFECV. Fit a new StandardScaler on `X_train_rfecv` and then transform `X_train_rfecv`, `X_val_rfecv`, and `X_test_rfecv`.


**Reasoning**:
To ensure that the features are on a similar scale for the Logistic Regression model, I will re-scale the RFECV selected features using StandardScaler. This helps in preventing features with larger values from dominating the learning process.



In [93]:
from sklearn.preprocessing import StandardScaler

# Initialize a new StandardScaler
scaler_rfecv = StandardScaler()

# Fit the scaler on X_train_rfecv and transform it
X_train_rfecv_scaled = scaler_rfecv.fit_transform(X_train_rfecv)

# Transform X_val_rfecv and X_test_rfecv using the fitted scaler
X_val_rfecv_scaled = scaler_rfecv.transform(X_val_rfecv)
X_test_rfecv_scaled = scaler_rfecv.transform(X_test_rfecv)

print("RFECV selected features successfully scaled.")
print(f"X_train_rfecv_scaled shape: {X_train_rfecv_scaled.shape}")
print(f"X_val_rfecv_scaled shape: {X_val_rfecv_scaled.shape}")
print(f"X_test_rfecv_scaled shape: {X_test_rfecv_scaled.shape}")

RFECV selected features successfully scaled.
X_train_rfecv_scaled shape: (3500, 15)
X_val_rfecv_scaled shape: (876, 15)
X_test_rfecv_scaled shape: (1450, 15)


## Re-train Logistic Regression Model

### Subtask:
Train a new Logistic Regression model using the re-scaled training data with the RFECV-selected features.


**Reasoning**:
To complete the subtask, I need to initialize a Logistic Regression model, train it with the RFECV-selected and re-scaled training data, and then generate predictions (both classes and probabilities) on the corresponding validation data.



In [94]:
from sklearn.linear_model import LogisticRegression

# Initialize a new Logistic Regression model with the specified parameters
model_rfecv = LogisticRegression(max_iter=1000, random_state=42)

# Train the model using the RFECV-selected and re-scaled training data
model_rfecv.fit(X_train_rfecv_scaled, y_train)

print("Logistic Regression model re-trained successfully with RFECV-selected and re-scaled features.")

# Make predictions on the RFECV-selected and re-scaled validation data
y_pred_rfecv_scaled = model_rfecv.predict(X_val_rfecv_scaled)
y_prob_rfecv_scaled = model_rfecv.predict_proba(X_val_rfecv_scaled)[:, 1]

print("Predictions on validation set with RFECV-selected features completed.")

Logistic Regression model re-trained successfully with RFECV-selected and re-scaled features.
Predictions on validation set with RFECV-selected features completed.


### Evaluate Performance with RFECV Selected Features

**Subtask:**
Evaluate the performance of the Logistic Regression model trained with RFECV-selected features on the validation set. Compare its accuracy and error rate with the previous models.

**Reasoning**:
To evaluate the performance of the Logistic Regression model with RFECV-selected features, I need to calculate the accuracy score and error rate on the validation set (`y_val`, `y_pred_rfecv_scaled`) and then compare it to the previously recorded original model's performance.



In [95]:
from sklearn.metrics import accuracy_score

accuracy_rfecv_scaled = accuracy_score(y_val, y_pred_rfecv_scaled)
error_rate_rfecv_scaled = (1 - accuracy_rfecv_scaled) * 100

print(f"Accuracy Score with RFECV and scaled features: {accuracy_rfecv_scaled:.4f}")
print(f"Error Rate with RFECV and scaled features: {error_rate_rfecv_scaled:.2f}%")

print("\n--- Comparison with Original and RFE Model ---")
print(f"Original Model Accuracy: {performance_summary['original_accuracy']:.4f}")
print(f"Original Model Error Rate: {performance_summary['original_error_rate']:.2f}%")
print(f"RFE (fixed 50 features) Model Accuracy: {performance_summary['rfe_accuracy']:.4f}")
print(f"RFE (fixed 50 features) Model Error Rate: {performance_summary['rfe_error_rate']:.2f}%")

# Update performance summary with RFECV results
performance_summary["rfecv_accuracy"] = accuracy_rfecv_scaled
performance_summary["rfecv_error_rate"] = error_rate_rfecv_scaled

Accuracy Score with RFECV and scaled features: 0.7808
Error Rate with RFECV and scaled features: 21.92%

--- Comparison with Original and RFE Model ---
Original Model Accuracy: 0.7808
Original Model Error Rate: 21.92%
RFE (fixed 50 features) Model Accuracy: 0.7808
RFE (fixed 50 features) Model Error Rate: 21.92%


### Performance Improvement Summary

- **Original Model Accuracy**: 0.7808
- **Original Model Error Rate**: 21.92%

- **RFE Model Accuracy**: 0.7808
- **RFE Model Error Rate**: 21.92%

- **RFECV Model Accuracy**: 0.7808
- **RFECV Model Error Rate**: 21.92%

In this instance, both RFE with `n_features_to_select=50` and RFECV for optimal feature selection did not lead to a change in the accuracy score or error rate on the validation set compared to the full feature set. While accuracy remained the same, both techniques successfully reduced the dimensionality of the feature space significantly (from 599 features to 50 for RFE, and to 15 for RFECV). This reduction can lead to benefits such as faster training times, reduced memory usage, and potentially better generalization on unseen data, even if the validation accuracy didn't immediately reflect an improvement. RFECV provided an even more parsimonious model with only 15 features while maintaining performance.

### Performance Improvement Summary

- **Original Model Accuracy**: 0.7808
- **Original Model Error Rate**: 21.92%

- **RFE Model Accuracy**: 0.7808
- **RFE Model Error Rate**: 21.92%

- **RFECV Model Accuracy**: 0.7808
- **RFECV Model Error Rate**: 21.92%

In this instance, both RFE with `n_features_to_select=50` and RFECV for optimal feature selection did not lead to a change in the accuracy score or error rate on the validation set compared to the full feature set. While accuracy remained the same, both techniques successfully reduced the dimensionality of the feature space significantly (from 599 features to 50 for RFE, and to 15 for RFECV). This reduction can lead to benefits such as faster training times, reduced memory usage, and potentially better generalization on unseen data, even if the validation accuracy didn't immediately reflect an improvement. RFECV provided an even more parsimonious model with only 15 features while maintaining performance.

## Make Predictions on Test Set (RFECV)

### Subtask:
Use the RFECV-optimized and re-trained Logistic Regression model to make predictions on the re-scaled test set with the selected features.


**Reasoning**:
To make predictions on the test set with the RFECV-optimized model, I will use the `predict` method for class labels and `predict_proba` for the positive class probabilities on the `X_test_rfecv_scaled` data.



In [96]:
y_test_pred_rfecv_scaled = model_rfecv.predict(X_test_rfecv_scaled)
y_test_prob_rfecv_scaled = model_rfecv.predict_proba(X_test_rfecv_scaled)[:, 1]

print("Predictions on the test set with RFECV-selected features completed successfully.")
print("Sample of predicted classes (first 10):", y_test_pred_rfecv_scaled[:10])
print("Sample of predicted probabilities for positive class (first 10):", y_test_prob_rfecv_scaled[:10])

Predictions on the test set with RFECV-selected features completed successfully.
Sample of predicted classes (first 10): [1 1 1 1 1 1 1 1 1 1]
Sample of predicted probabilities for positive class (first 10): [0.70197469 0.93415125 0.8105821  0.87729595 0.80592113 0.86796671
 0.81147229 0.73744771 0.80095034 0.89476623]


**Reasoning**:
To finalize the predictions for submission, I will create a pandas DataFrame containing the 'customerID' from the `test_ids` and the `y_test_pred_rfecv_scaled` predictions, which represents the 'Good_Bad_flag'.



In [97]:
submission_rfecv = pd.DataFrame({
    'customerID': test_ids,
    'Good_Bad_flag': y_test_pred_rfecv_scaled
})

print("Submission preview with RFECV-selected features:")
print(submission_rfecv.head())


Submission preview with RFECV-selected features:
                         customerID  Good_Bad_flag
0  8a858899538ddb8e015390510b321f08              1
1  8a858959537a097401537a4e316e25f7              1
2  8a8589c253ace09b0153af6ba58f1f31              1
3  8a858e095aae82b7015aae86ca1e030b              1
4  8a858e225a28c713015a30db5c48383d              1


In [ ]:
submission.to_csv("submission.csv", index=False)

## Generate Submission File (RFECV)

### Subtask:
Create a submission file in CSV format, containing the customer IDs and the binary predictions (1 for 'Good', 0 for 'Bad') from the RFECV-optimized Logistic Regression model.


**Reasoning**:
To finalize the submission, I will save the `submission_rfecv` DataFrame to a CSV file as requested, ensuring that the index is not included in the output.



In [98]:
submission_rfecv.to_csv("submission_rfecv.csv", index=False)

print("Submission file 'submission_rfecv.csv' created successfully.")

Submission file 'submission_rfecv.csv' created successfully.


## Final Task

### Subtask:
Summarize the optimization process using RFECV and the performance of the Logistic Regression model, highlighting any improvements achieved.


## Summary:

### Q&A
The optimization process using RFECV significantly reduced the number of features while maintaining the Logistic Regression model's performance. The original model used 599 features and achieved an accuracy of 0.7808 with an error rate of 21.92%. The RFE model with a fixed 50 features also yielded the same accuracy and error rate. The RFECV process identified an optimal set of 15 features, and when a Logistic Regression model was trained and evaluated on these 15 features, it achieved an accuracy of 0.7808 and an error rate of 21.92% on the validation set. This indicates that while no improvement in validation accuracy was observed, the model became significantly more parsimonious, reducing the feature space by 97.5% (from 599 to 15 features). This reduction can lead to benefits such as faster training, reduced memory usage, and improved interpretability.

### Data Analysis Key Findings
*   Recursive Feature Elimination with Cross-Validation (RFECV) successfully identified an optimal set of 15 features, a significant reduction from the original 599 features.
*   The training, validation, and test datasets were successfully transformed to retain only these 15 features, changing their shape from `(N, 599)` to `(N, 15)`.
*   A new `StandardScaler` was fitted on the RFECV-selected training features and used to re-scale all datasets (training, validation, and test).
*   A Logistic Regression model re-trained on the 15 RFECV-selected and re-scaled features achieved an accuracy of 0.7808 and an error rate of 21.92% on the validation set.
*   This performance (accuracy and error rate) was identical to that of the original Logistic Regression model trained on all 599 features and the RFE model trained on 50 fixed features.
*   Predictions were successfully made on the test set using the RFECV-optimized model, and a submission file `submission_rfecv.csv` was generated.

### Insights or Next Steps
*   The drastic reduction in feature count (from 599 to 15) without any loss in validation accuracy makes the model significantly more efficient, reducing computational overhead and potentially improving interpretability, which is a valuable optimization.
*   Further exploration could involve evaluating the model's performance on the test set (which will be done implicitly during submission), or investigating if more complex models could leverage this optimized feature set to achieve even higher predictive accuracy.
